In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import MDAnalysis as mda
import numpy as np
from MDAnalysisTests.datafiles import DCD, PSF

from mdadash.backend.analyses.rog import ROG
from mdadash.backend.kernel.core import BufferedTrajectory

u = mda.Universe(PSF, DCD)

buffer_size = 50
u.trajectory = BufferedTrajectory(u.trajectory, buffer_size, use_shm=False)

r = ROG()
r.u = u
# vectorization is only supported in batch mode if trajectory is already
# unwrapped. Disable unwrap to ensure we comparing correctly
r.unwrap = False
r.selection = "protein"
r.on_post_connect()
for i in range(buffer_size - 1):
    u.trajectory._prepare_for_next()
    u.trajectory.next()

In [3]:
# without vectorization
r.batch_vectorization = False
non_vec_values = r._compute_batch()
%timeit r._compute_batch()

16.6 ms ± 1 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
# with vectorization
r.batch_vectorization = True
vec_values = r._compute_batch()
%timeit r._compute_batch()

2.3 ms ± 74 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# ensure computed values are the same
non_vec_rog = np.array([item[2] for item in non_vec_values])
vec_rog = np.array([item[2] for item in vec_values])
np.allclose(non_vec_rog, vec_rog, rtol=1e-05, atol=1e-05)

True

In [ ]:
r._reset_plot_values()
r._update_plot(non_vec_values)

In [ ]:
r._reset_plot_values()
r._update_plot(vec_values)